<a href="https://colab.research.google.com/github/tantg88888/sit-ai-practice-2026/blob/main/week1-hdb-resale-price-prediction/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HDB Resale Price Prediction: valuation model + Gradio app

Predicts the resale price (SGD) of a Singapore HDB flat from its town, street, flat type and model, storey band, floor area, lease start year and transaction month. It uses a leakage-safe scikit-learn pipeline (one-hot + per-fold target encoding + log target) with a tuned HistGradientBoosting champion. It was chosen with time-series cross-validation and scored once on a future hold-out period (2025-01 → 2026-06, R² 0.903 in the recorded run). The notebook follows CRISP-DM and includes ablations, failure analysis and a lender/buyer safety-margin decision. It ends with an interactive **HDB valuation** app.

**▶ Demo: Runtime → Run all (TRAIN = False). The Gradio link appears at the bottom in about 3–5 min.**
- Runtime: **CPU** is enough (no GPU needed).
- Demo mode skips the slow cross-validation, ablation and tuning cells and shows their **recorded** results. The final model is retrained live, which takes about a minute.
- Set `TRAIN = True` in the first code cell to reproduce the full run (about 5–10 min on free Colab CPU).

**Team credit:** group capstone (Group 7), ICT6001 Week 1.


# HDB Resale Flat Price Prediction — Capstone Solution

**Course:** ICT6001 Week01 Group Capstone  &nbsp;|&nbsp; **Group:** Group 7
**Task type:** Supervised **regression** — predict `resale_price` (SGD)
**Dataset:** `hdb_resale_flat_prices.csv` — 233,265 transactions, Jan 2017 → Jun 2026 (course snapshot)

This notebook follows the CRISP-DM flow required by the spec and is organised into the five graded parts:

| Part | Section | Rubric criterion it targets |
|---|---|---|
| A | Advanced Data Prep & Pipeline Engineering | Pipeline Engineering & Data Prep |
| B | Champion Model Selection | Model Selection & Tuning Discipline |
| C | Controlled Ablations & Tuning | Model Selection & Tuning Discipline |
| D | Mechanical Failure Analysis | Failure Analysis |
| E | Business Decision Reasoning | Business Decision Reasoning |

> **Design principle:** every learned transformation lives *inside* an sklearn `Pipeline` and is fit on training folds only. Row-level deterministic feature engineering (string parsing, arithmetic) is done before the split because it learns no parameters from the data — this distinction is what prevents leakage.

> **Runtime note:** the full run (TRAIN = True) is about 5–10 minutes on a free Colab CPU (the tree-ensemble cross-validation on ~197k training rows dominates). All compute settings are deliberately modest, in line with the spec's instruction *not* to brute-force.

## Stage 1 — Business & Data Understanding (CRISP-DM Phase 1)

**Stakeholder & use case.** An automated valuation model (AVM) for a bank's housing-loan desk (and, secondarily, for buyers gauging whether a listing is fairly priced). The model outputs an estimated resale price that informs (a) buyer advice and (b) the loan amount underwritten against the flat as collateral.

**Cost of errors (set now, used in Part E).**
- **Over-prediction** (estimate > true value): the bank over-lends against collateral that won't cover the loan if the borrower defaults — a direct credit-loss risk. This is the *more damaging* error for the lender.
- **Under-prediction** (estimate < true value): the buyer is advised conservatively / the loan is smaller than it could be — mostly a lost-opportunity cost, less damaging.

**Success criteria.** Low dollar error (RMSE/MAE) on a *future* time period, because the model will always predict forward from historical transactions. Stability across time matters as much as the headline number.

## 0. Environment Setup

In [1]:
# ── Demo settings ──────────────────────────────────────────────
# TRAIN = False: interview demo. Skips the slow CV / ablation / tuning cells (their recorded results are shown instead),
#                retrains the final model (~1 min on CPU) and launches the Gradio valuation app.
# TRAIN = True : reproduces the full original run (model race, 4 ablations, tuning search) - about 5-10 min on CPU.
TRAIN = False

from IPython.core.magic import register_cell_magic
@register_cell_magic
def train_only(line, cell):
    """Cells marked %%train_only run only when TRAIN = True."""
    if TRAIN:
        get_ipython().run_cell(cell)
    else:
        print("⏭ Skipped in demo mode (set TRAIN = True to run this cell).")


In [2]:
!pip install -q category_encoders gradio

In [3]:
import warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from category_encoders import TargetEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("Environment ready.")

Environment ready.


### 1.1 Load data, integrity checks

In [4]:
# Load the data. Sources are tried in order:
#   1) a local copy (hdb_resale_flat_prices.csv or data/hdb_resale_flat_prices.csv)
#   2) the course snapshot hosted in this project's GitHub repo (Jan 2017 -> Jun 2026, the data the recorded results used)
#   3) the live public dataset from data.gov.sg ("Resale flat prices based on registration date from Jan-2017 onwards")
import os, time, requests

CSV_NAME   = "hdb_resale_flat_prices.csv"
GITHUB_RAW = ("https://raw.githubusercontent.com/tantg88888/sit-ai-practice-2026/main/"
              "week1-hdb-resale-price-prediction/data/hdb_resale_flat_prices.csv")
DATASET_ID = "d_8b84c4ee58e3cfc0ece0d773c8ca6abc"
DATAGOV    = f"https://api-open.data.gov.sg/v1/public/api/datasets/{DATASET_ID}"
COLUMNS    = ["month", "town", "flat_type", "block", "street_name", "storey_range", "floor_area_sqm",
              "flat_model", "lease_commence_date", "remaining_lease", "resale_price"]

def _csv_url(resp):
    """Extract data.url from a data.gov.sg JSON response (None if absent or not JSON)."""
    try:
        return ((resp.json() or {}).get("data") or {}).get("url")
    except ValueError:
        return None

def _download_from_datagov(max_tries=12):
    """data.gov.sg v1 API: poll-download returns a temporary CSV URL once the file is ready;
    initiate-download asks the server to prepare it."""
    initiated = False
    for attempt in range(max_tries):
        r = requests.get(f"{DATAGOV}/poll-download", timeout=30)
        if r.status_code == 429:                       # rate limited -> wait and retry
            time.sleep(5); continue
        url = _csv_url(r)
        if url:
            return pd.read_csv(url)
        if not initiated:                              # file not prepared yet -> ask for it once
            ri = requests.get(f"{DATAGOV}/initiate-download", timeout=30)
            if ri.status_code == 405:
                ri = requests.post(f"{DATAGOV}/initiate-download", timeout=30)
            initiated = True
            url = _csv_url(ri)
            if url:
                return pd.read_csv(url)
        time.sleep(3)
    raise RuntimeError("data.gov.sg download was not ready in time - re-run this cell.")

df, DATA_SOURCE = None, None
for p in [CSV_NAME, os.path.join("data", CSV_NAME)]:
    if os.path.exists(p):
        df, DATA_SOURCE = pd.read_csv(p), f"local file ({p})"
        break
if df is None:
    try:
        df, DATA_SOURCE = pd.read_csv(GITHUB_RAW), "GitHub copy of the course snapshot"
    except Exception as e:
        print("GitHub copy not available:", type(e).__name__)
if df is None:
    df, DATA_SOURCE = _download_from_datagov(), "data.gov.sg live API"
    print("NOTE: live data may include months after Jun 2026, so every number below can differ "
          "slightly from the recorded run (the hold-out simply covers 2025-01 up to the latest month).")
    df.to_csv(CSV_NAME, index=False)                   # cache for re-runs

df = df[COLUMNS]
df["remaining_lease"] = df["remaining_lease"].astype(str)
print("Data source:", DATA_SOURCE)
print("Raw shape:", df.shape, f"({df.month.min()} .. {df.month.max()})")
print("\nMissing values per column:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
df.head()


GitHub copy not available: HTTPError
NOTE: live data may include months after Jun 2026, so every number below can differ slightly from the recorded run (the hold-out simply covers 2025-01 up to the latest month).
Data source: data.gov.sg live API
Raw shape: (243500, 11) (2017-01 .. 2026-10)

Missing values per column:
 month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
dtype: int64

Exact duplicate rows: 318


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.00,Improved,1979,61 years 04 months,"232,000.00"
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.00,New Generation,1978,60 years 07 months,"250,000.00"
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.00,New Generation,1980,62 years 05 months,"262,000.00"
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.00,New Generation,1980,62 years 01 month,"265,000.00"
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.00,New Generation,1980,62 years 05 months,"265,000.00"


### 1.2 Categorical cardinality audit → encoding strategy

In [5]:
cats = ["town", "flat_type", "flat_model", "storey_range", "street_name", "block"]
print(df[cats].nunique().sort_values().to_string())

flat_type          7
storey_range      17
flat_model        21
town              26
street_name      580
block           2789


**Decision.** Low-cardinality fields (`flat_type` 7, `flat_model` 21, `town` 26) → **one-hot**. The high-cardinality `street_name` (577) is too granular for one-hot but carries strong location signal → **target encoding**, fit *inside* the pipeline on training folds only (fitting it on the full set is the textbook leak). `block` (2,763) is dropped as too sparse.

## Part A — Advanced Data Preparation & Pipeline Engineering

Targets the **Pipeline Engineering & Data Prep** criterion. Goal: a single leakage-safe `Pipeline` that every model will share, so Part B is a "fair race".

### A.1 Cleaning (leakage-safe, row-level)

In [6]:
n_dup = int(df.duplicated().sum())
data = df.drop_duplicates().reset_index(drop=True)
print(f"After dropping {n_dup:,} duplicates:", data.shape)
# Sanity bounds (no rows learned-from; this is not leakage)
print("floor_area_sqm:", data.floor_area_sqm.min(), "-", data.floor_area_sqm.max())
print("resale_price  :", int(data.resale_price.min()), "-", int(data.resale_price.max()))


After dropping 318 duplicates: (243182, 11)
floor_area_sqm: 31.0 - 366.7
resale_price  : 140000 - 1728000


### A.2 Feature engineering (deterministic → safe before split)

These transforms parse existing fields with fixed rules; they fit no parameters, so applying them globally introduces no leakage.
- `remaining_lease_months` — parse "X years Y months" → integer months (99-yr leasehold decay).
- `storey_mid` — band "10 TO 12" → midpoint 11 (preserves ordinal height premium).
- `flat_age` = transaction year − `lease_commence_date`.
- `txn_year`, `txn_month_num` — from `month`. `txn_year` is also the split key.

> **Leakage trap avoided:** we do **not** build `price_per_sqm` as a feature — it contains the target and would leak.

In [7]:
def engineer(frame):
    out = frame.copy()
    def lease_to_months(s):
        y = re.search(r"(\d+)\s*year", s); m = re.search(r"(\d+)\s*month", s)
        return (int(y.group(1)) if y else 0) * 12 + (int(m.group(1)) if m else 0)
    out["remaining_lease_months"] = out["remaining_lease"].map(lease_to_months)
    out["storey_mid"] = out["storey_range"].map(lambda s: (int(s[:2]) + int(s[-2:])) / 2)
    out["txn_date"] = pd.to_datetime(out["month"])
    out["txn_year"] = out["txn_date"].dt.year
    out["txn_month_num"] = out["txn_date"].dt.month
    out["flat_age"] = out["txn_year"] - out["lease_commence_date"]
    return out

data = engineer(data)
data[["remaining_lease_months", "storey_mid", "flat_age", "txn_year"]].describe().round(1)

,remaining_lease_months,storey_mid,flat_age,txn_year
count,"243,182.00","243,182.00","243,182.00","243,182.00"
mean,888.40,8.80,25.00,"2,021.60"
std,172.20,6.00,14.30,2.80
min,469.00,2.00,1.00,"2,017.00"
25%,745.00,5.00,11.00,"2,019.00"
50%,885.00,8.00,25.00,"2,022.00"
75%,"1,062.00",11.00,37.00,"2,024.00"
max,"1,173.00",50.00,60.00,"2,026.00"


### A.3 Feature lists and target

In [8]:
NUMERIC     = ["floor_area_sqm", "remaining_lease_months", "storey_mid", "flat_age", "txn_year", "txn_month_num"]
LOW_CARD    = ["flat_type", "town", "flat_model"]
HIGH_CARD   = ["street_name"]
FEATURES    = NUMERIC + LOW_CARD + HIGH_CARD
TARGET      = "resale_price"
print(f"{len(FEATURES)} features:", FEATURES)

10 features: ['floor_area_sqm', 'remaining_lease_months', 'storey_mid', 'flat_age', 'txn_year', 'txn_month_num', 'flat_type', 'town', 'flat_model', 'street_name']


### A.4 Time-based split — and **freeze** the test set

Justified by the +54% price trend: a random split leaks future price levels. We train on transactions up to **2024-12** and hold out **2025-01 → 2026-06** as the deployment-like future. The test set is touched exactly **once**, in Part E.

In [9]:
train_df = data[data["txn_date"] <= "2024-12"].sort_values("txn_date").reset_index(drop=True)
test_df  = data[data["txn_date"] >= "2025-01"].reset_index(drop=True)

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]
print(f"TRAIN {X_train.shape}  ({train_df.month.min()} .. {train_df.month.max()})")
print(f"TEST  {X_test.shape}  ({test_df.month.min()} .. {test_df.month.max()})  <-- FROZEN until Part E")

TRAIN (196686, 10)  (2017-01 .. 2024-12)
TEST  (46496, 10)  (2025-01 .. 2026-10)  <-- FROZEN until Part E


### A.5 The leakage-safe preprocessing pipeline

A `ColumnTransformer` with three branches, wrapped with the estimator in a `Pipeline`, and the whole thing wrapped in `TransformedTargetRegressor` (log target). `make_model()` is parameterised so Part C can toggle individual pieces for ablations without ever leaving the pipeline.

In [10]:
def make_preprocessor(numeric=NUMERIC, low_card=LOW_CARD, high_card=HIGH_CARD):
    branches = [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), numeric),
        ("low", OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_card),
    ]
    if high_card:                                   # target encoding fit per-fold (no leakage)
        branches.append(("high", TargetEncoder(), high_card))
    return ColumnTransformer(branches)

def make_model(estimator, numeric=NUMERIC, low_card=LOW_CARD, high_card=HIGH_CARD, log_target=True):
    pipe = Pipeline([("pre", make_preprocessor(numeric, low_card, high_card)),
                     ("model", estimator)])
    if log_target:
        return TransformedTargetRegressor(pipe, func=np.log1p, inverse_func=np.expm1)
    return pipe

make_model(Ridge())   # display the pipeline diagram

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('pre',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       Pipeline(steps=[('impute',
                                                                                                        SimpleImputer(strategy='median')),
                                                                                                       ('scale',
                                                                                                        StandardScaler())]),
                                                                                       ['floor_area_sqm',
                                                                                        'remaining_lease_months',
                                                                                        'storey_mid',
                                                                                        'flat_age',
                                                                                        'txn_year',
                                                                                        'txn_month_num']),
                                                                                      ('low',
                                                                                       OneHotEncoder(handle_unknown='ignore',
                                                                                                     sparse_output=False),
                                                                                       ['flat_type',
                                                                                        'town',
                                                                                        'flat_model']),
                                                                                      ('high',
                                                                                       TargetEncoder(),
                                                                                       ['street_name'])])),
                                                     ('model', Ridge())]))

### A.6 Pipeline sanity check (fit on a slice, confirm it predicts in dollars)

In [11]:
_demo = make_model(Ridge()).fit(X_train.iloc[:5000], y_train.iloc[:5000])
print("Sample predictions ($):", np.round(_demo.predict(X_test.iloc[:3])).tolist())
print("Actuals             ($):", y_test.iloc[:3].tolist())

Sample predictions ($): [206599.0, 211497.0, 218527.0]
Actuals             ($): [340000.0, 353000.0, 318000.0]


## Part B — Champion Model Selection

Targets **Model Selection & Tuning Discipline**. Three distinct algorithmic families race under the *same* preprocessing pipeline, evaluated with time-aware cross-validation.

### B.1 Primary metric & CV scheme
- **Primary:** RMSE in dollars — penalises the large valuation misses that carry the most credit risk.
- **Secondary:** MAE (typical dollar error) and R².
- **CV:** `TimeSeriesSplit(5)` on the (date-sorted) training set — never trains on the future to score the past, consistent with the held-out future test set.

In [12]:
cv = TimeSeriesSplit(n_splits=5)
scoring = {"rmse": "neg_root_mean_squared_error",
           "mae":  "neg_mean_absolute_error",
           "r2":   "r2"}

candidates = {
    "Ridge (linear)":        Ridge(alpha=1.0),
    "RandomForest (bagging)": RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                                                    n_jobs=-1, random_state=RANDOM_STATE),
    "HistGB (boosting)":      HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1,
                                                            random_state=RANDOM_STATE),
}

### B.2 Cross-validate the candidates (~2-3 min)

In [13]:
%%train_only
rows = []
for name, est in candidates.items():
    s = cross_validate(make_model(est), X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({
        "Model": name,
        "RMSE mean": -s["test_rmse"].mean(), "RMSE std": s["test_rmse"].std(),
        "MAE mean":  -s["test_mae"].mean(),
        "R2 mean":   s["test_r2"].mean(),
    })
cv_results = pd.DataFrame(rows).sort_values("RMSE mean").reset_index(drop=True)
cv_results.round(3)

,Model,RMSE mean,RMSE std,MAE mean,R2 mean
0,HistGB (boosting),"52,839.74","7,419.57","38,212.14",0.90
1,RandomForest (bagging),"62,185.92","9,251.97","42,521.94",0.86
2,Ridge (linear),"72,330.81","10,925.50","54,208.07",0.81


In [14]:
# Demo mode: the 3-model x 5-fold race above was skipped, so show the RECORDED results from the full training run.
if not TRAIN:
    from IPython.display import display
    cv_results = pd.DataFrame(
        [["HistGB (boosting)",      52839.74,  7419.57, 38212.14, 0.90],
         ["RandomForest (bagging)", 62185.92,  9251.97, 42521.94, 0.86],
         ["Ridge (linear)",         72330.81, 10925.50, 54208.07, 0.81]],
        columns=["Model", "RMSE mean", "RMSE std", "MAE mean", "R2 mean"])
    print("Recorded from the full training run (TRAIN = True), not recomputed in demo mode:")
    display(cv_results.round(3))


### B.3 Declare the Champion

Expected ordering (your numbers will be close): **HistGB** wins on both lower mean RMSE *and* lower std (most stable across folds), comfortably ahead of RandomForest, with Ridge as the interpretable floor. We therefore declare **HistGradientBoosting the Champion** and carry it alone into Part C. Ridge is retained only as a sanity reference.

In [15]:
champion_name = cv_results.iloc[0]["Model"]
print("CHAMPION:", champion_name)

CHAMPION: HistGB (boosting)


## Part C — Controlled Ablations & Tuning (Champion only)

Targets **Model Selection & Tuning Discipline**. Four controlled experiments, **one variable changed each**, all on training-set CV (test set stays frozen). Tuning budget is held small on purpose (`RandomizedSearchCV`, n_iter=8, cv=3 → 24 fits, well under the 50-iteration cap).

In [16]:
# Helpers used by the experiments below and by Parts D/E (defined in both demo and full mode)
def cv_rmse(model, X, y, splits=3):
    s = cross_validate(model, X, y, cv=TimeSeriesSplit(splits),
                       scoring="neg_root_mean_squared_error", n_jobs=1)
    return -s["test_score"].mean(), s["test_score"].std()

HGB = lambda **kw: HistGradientBoostingRegressor(random_state=RANDOM_STATE, **kw)
ablation = []


In [17]:
%%train_only
# Baseline champion (modest settings, 3-fold for speed/consistency across experiments)
base_mean, base_std = cv_rmse(make_model(HGB(max_iter=150)), X_train, y_train)
ablation.append(["Baseline (HistGB, target-enc, log target)", "—", base_mean, base_std, "Reference"])
print("baseline done")


baseline done


#### Experiment 1 — value of target encoding (drop the high-cardinality location branch)

In [18]:
%%train_only
m1, s1 = cv_rmse(make_model(HGB(max_iter=150), high_card=[]),
                 X_train.drop(columns=HIGH_CARD), y_train)
ablation.append(["1. Remove target-encoded street_name",
                 "high_card branch dropped", m1, s1,
                 "Reject removal — RMSE rises, so target encoding adds location signal"])
print(f"Exp1 RMSE {m1:,.0f} vs base {base_mean:,.0f}")

Exp1 RMSE 68,606 vs base 66,906


#### Experiment 2 — value of the lease-decay feature

In [19]:
%%train_only
no_lease = [c for c in NUMERIC if c != "remaining_lease_months"]
m2, s2 = cv_rmse(make_model(HGB(max_iter=150), numeric=no_lease),
                 X_train.drop(columns=["remaining_lease_months"]), y_train)
ablation.append(["2. Remove remaining_lease_months",
                 "lease feature dropped", m2, s2,
                 "Keep feature — removing it gives a near-identical RMSE (effectively a tie, "
                 "flat_age already carries most of the age signal); kept for its domain meaning (99-year lease decay)"])
print(f"Exp2 RMSE {m2:,.0f} vs base {base_mean:,.0f}")


Exp2 RMSE 66,945 vs base 66,906


#### Experiment 3 — hyperparameter tuning (≤3 params, n_iter=8, cv=3 → 24 fits)

In [20]:
%%train_only
search = RandomizedSearchCV(
    make_model(HGB()),
    param_distributions={
        "regressor__model__learning_rate": [0.05, 0.1, 0.2],
        "regressor__model__max_iter":      [150, 300, 500],
        "regressor__model__max_leaf_nodes":[31, 63, 127],
    },
    n_iter=8, cv=TimeSeriesSplit(3),
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE, n_jobs=1,
)
search.fit(X_train, y_train)
best = {k.split("__")[-1]: v for k, v in search.best_params_.items()}
ablation.append(["3. Tune learning_rate / max_iter / max_leaf_nodes",
                 f"RandomizedSearch (8 iters): {best}", -search.best_score_,
                 search.cv_results_["std_test_score"][search.best_index_],
                 "Accept — best config lowers RMSE; adopt as refined champion"])
print("Best params:", best, " RMSE:", round(-search.best_score_))

Best params: {'max_leaf_nodes': 31, 'max_iter': 300, 'learning_rate': 0.2}  RMSE: 60799


#### Experiment 4 — raw target vs log target

**Honest result (recorded run):** the raw target scored slightly *better* on 3-fold CV than the log-target baseline (RMSE 65,924 vs 66,906, about 1.5% lower). That gap is far smaller than the fold-to-fold spread (CV std about 11,000), so it is not a clear win. We kept the log target for two reasons. First, the tuned configuration from Experiment 3 was searched on the log target. Second, log(price) compresses the sparse million-dollar tail, where the worst errors sit (Part D). We did not re-test the raw target with the tuned settings; that is a follow-up.


In [21]:
%%train_only
m4, s4 = cv_rmse(make_model(HGB(max_iter=150), log_target=False), X_train, y_train)
if m4 < base_mean:
    exp4_conclusion = ("Keep log target despite a slightly better raw-target CV score (about "
                       f"{(base_mean - m4) / base_mean:.1%} lower RMSE, well inside one CV std): the tuning search "
                       "(Exp 3) was run on the log target, and log(price) compresses the sparse million-dollar tail (Part D). "
                       "Re-testing raw target with the tuned settings is an open follow-up.")
else:
    exp4_conclusion = "Reject — raw target worsens RMSE, confirming the log-target decision"
ablation.append(["4. Raw target instead of log target",
                 "log_target=False", m4, s4, exp4_conclusion])
print(f"Exp4 RMSE {m4:,.0f} vs base {base_mean:,.0f}")


Exp4 RMSE 65,924 vs base 66,906


### Consolidated Ablation Log

In [22]:
%%train_only
ablation_log = pd.DataFrame(ablation, columns=["Experiment", "Controlled Change",
                                               "CV RMSE mean", "CV RMSE std", "Conclusion"])
ablation_log

,Experiment,Controlled Change,CV RMSE mean,CV RMSE std,Conclusion
0,"Baseline (HistGB, target-enc, log target)",—,"66,906.17","10,939.46",Reference
1,1. Remove target-encoded street_name,high_card branch dropped,"68,606.13","12,313.67","Reject removal — RMSE rises, so target encodin..."
2,2. Remove remaining_lease_months,lease feature dropped,"66,944.58","11,163.48",Keep feature — removing it gives a near-identi...
3,3. Tune learning_rate / max_iter / max_leaf_nodes,RandomizedSearch (8 iters): {'max_leaf_nodes':...,"60,799.11","10,157.92",Accept — best config lowers RMSE; adopt as ref...
4,4. Raw target instead of log target,log_target=False,"65,924.34","11,165.36",Keep log target despite a slightly better raw-...


In [23]:
# Demo mode: Experiments 1-4 above were skipped, so use the RECORDED results from the full training run.
if not TRAIN:
    from IPython.display import display
    best = {'max_leaf_nodes': 31, 'max_iter': 300, 'learning_rate': 0.2}   # Exp 3 RandomizedSearchCV winner
    ablation_log = pd.DataFrame([
        ["Baseline (HistGB, target-enc, log target)", "—", 66906.17, 10939.46, "Reference"],
        ["1. Remove target-encoded street_name", "high_card branch dropped", 68606.13, 12313.67,
         "Reject removal — RMSE rises, so target encoding adds location signal"],
        ["2. Remove remaining_lease_months", "lease feature dropped", 66944.58, 11163.48,
         "Keep feature — removing it gives a near-identical RMSE (effectively a tie, flat_age already carries "
         "most of the age signal); kept for its domain meaning (99-year lease decay)"],
        ["3. Tune learning_rate / max_iter / max_leaf_nodes", f"RandomizedSearch (8 iters): {best}", 60799.11, 10157.92,
         "Accept — best config lowers RMSE; adopt as refined champion"],
        ["4. Raw target instead of log target", "log_target=False", 65924.34, 11165.36,
         "Keep log target despite a slightly better raw-target CV score (about 1.5% lower RMSE, well inside one CV std): "
         "the tuning search (Exp 3) was run on the log target, and log(price) compresses the sparse million-dollar tail "
         "(Part D). Re-testing raw target with the tuned settings is an open follow-up."],
    ], columns=["Experiment", "Controlled Change", "CV RMSE mean", "CV RMSE std", "Conclusion"])
    print("Recorded from the full training run (TRAIN = True), not recomputed in demo mode:")
    display(ablation_log)


In [24]:
# Lock the refined champion configuration from Experiment 3
refined_champion = make_model(HGB(**best))
print("Refined champion locked:", best)

Refined champion locked: {'max_leaf_nodes': 31, 'max_iter': 300, 'learning_rate': 0.2}


## Part D — Mechanical Failure Analysis

Targets **Failure Analysis**. We inspect the *rows* the champion gets most wrong (highest absolute error) on a leakage-safe validation slice, then explain the mechanism and propose fixes — not aggregate metrics.

### D.1 Leakage-safe validation predictions (time-consistent)

`cross_val_predict` cannot be used with `TimeSeriesSplit` (it is not a partition — early rows never appear in a test fold). To stay time-consistent *and* leakage-safe we carve an explicit forward-validation slice from the **training** data: fit on transactions up to 2023-12, predict the 2024 slice. The 2025–2026 test set stays frozen.

In [25]:
fit_mask = train_df["txn_date"] <= "2023-12"
val_mask = train_df["txn_date"] >= "2024-01"     # still inside the training window
diag_model = make_model(HGB(max_iter=150)).fit(X_train[fit_mask], y_train[fit_mask])

diag = train_df[val_mask].copy()
diag["pred"] = diag_model.predict(X_train[val_mask])
diag["error"] = diag["pred"] - diag[TARGET]      # +ve = over-predict, -ve = under-predict
diag["abs_error"] = diag["error"].abs()
print(f"Validation rows (2024): {len(diag):,}")
print(f"Validation MAE: ${diag.abs_error.mean():,.0f}   |   worst abs error: ${diag.abs_error.max():,.0f}")

Validation rows (2024): 27,817
Validation MAE: $49,740   |   worst abs error: $564,364


### D.2 The 8 highest-absolute-error cases

In [26]:
cols = ["town", "flat_type", "flat_model", "floor_area_sqm", "storey_mid",
        "remaining_lease_months", TARGET, "pred", "error"]
worst = diag.nlargest(8, "abs_error")[cols]
worst.round(0)

,town,flat_type,flat_model,floor_area_sqm,storey_mid,remaining_lease_months,resale_price,pred,error
182367,ANG MO KIO,5 ROOM,Adjoined flat,134.00,8.00,644,"1,260,000.00","695,636.00","-564,364.00"
196018,TOA PAYOH,5 ROOM,Improved,114.00,11.00,1138,"1,340,000.00","861,948.00","-478,052.00"
188334,BUKIT MERAH,5 ROOM,Improved,113.00,41.00,1050,"1,580,000.00","1,120,695.00","-459,305.00"
194112,TOA PAYOH,5 ROOM,Improved,114.00,8.00,1138,"1,310,000.00","856,330.00","-453,670.00"
189505,ANG MO KIO,5 ROOM,Improved,121.00,11.00,1045,"1,250,000.00","813,285.00","-436,715.00"
194072,QUEENSTOWN,5 ROOM,Improved,124.00,38.00,1037,"1,550,000.00","1,116,517.00","-433,483.00"
190922,TOA PAYOH,5 ROOM,Improved,113.00,14.00,1139,"1,300,000.00","879,476.00","-420,524.00"
190659,ANG MO KIO,5 ROOM,Improved,121.00,20.00,1044,"1,300,000.00","882,451.00","-417,549.00"


### D.3 Mechanical explanation

The worst errors are almost all **large under-predictions of million-dollar flats** (e.g. a 150 m² Adjoined-flat in a central town selling far above what its features imply). The mechanism:

1. **Sparse luxury tail.** Million-dollar transactions are rare and recent; the tree leaves covering big/old/central units hold few high-price examples, so predictions regress toward the populated mid-market — a systematic *under*-shoot at the top.
2. **Rare categories.** `MULTI-GENERATION` (86 rows) and `1 ROOM` (87 rows) have too few samples to learn stable leaf values (per-segment MAE confirms error grows with flat size/rarity).
3. **Unmodelled premium.** A specific block/street commanding a heritage or location premium beyond what `town`-level encoding captures.

In [27]:
seg = (diag.assign(abs_error=diag.abs_error)
            .groupby("flat_type")["abs_error"].mean().sort_values())
seg.round(0)

,abs_error
flat_type,
1 ROOM,"13,685.00"
2 ROOM,"22,349.00"
3 ROOM,"35,962.00"
4 ROOM,"51,963.00"
5 ROOM,"58,301.00"
EXECUTIVE,"68,687.00"
MULTI-GENERATION,"108,519.00"


### D.4 Proposed technical fixes
1. **Compress the tail** — already modelling `log(price)`; additionally consider winsorising the top 1% or adding a "luxury" indicator (floor_area > 95th pct) so the model can branch on it.
2. **Engineer interactions** — `town × flat_type` or `floor_area × storey_mid` to give premium combinations their own signal instead of an averaged leaf.
3. **Handle rare segments** — pool `MULTI-GENERATION`/`1 ROOM` into nearest types, or attach a low-confidence flag and route these to manual valuation.
4. **Finer location** — add planning-area or postal-sector features so block-level premia are captured without the sparsity of raw `street_name`.

## Part E — Business Decision Reasoning

Targets **Business Decision Reasoning**. For regression the decision is a **safety margin**, not a probability threshold.

### E.1 Direction of the model's error in the deployment period

In [28]:
# Validation-period (2024) bias direction
print(f"Validation mean signed error: ${diag.error.mean():,.0f}")
print("Over-predicted: %.1f%%   Under-predicted: %.1f%%" %
      ((diag.error > 0).mean()*100, (diag.error < 0).mean()*100))

Validation mean signed error: $-45,449
Over-predicted: 12.5%   Under-predicted: 87.5%


### E.2 Decision: which error is costlier, and the margin

From **Stage 1**, the costlier error for the *lender* is **over-prediction** (over-lending against weak collateral). Empirically (Part D and above, and confirmed on the held-out future below) the time-split model in a rising market **already under-predicts** — its natural bias is conservative, which happens to align with lender prudence.

**Recommendation:**
- For the **lending** use case, *keep* the conservative bias — apply at most a small negative safety margin to the few over-predicted cases; do **not** correct the under-prediction upward, as that would import credit risk.
- For the **buyer-advice** use case, the systematic under-valuation is unfair to sellers, so a modest positive correction (e.g. +3–5%, or a recency-trend adjustment) is justified there.

This asymmetry — *same model, opposite margin by use case, driven by the Stage 1 cost structure* — is the core business judgement.

### E.3 FINAL single hold-out evaluation

The test set is now unfrozen for the **one and only** time. We fit the refined champion on **all** training data and predict the 2025–2026 future once.

In [29]:
refined_champion.fit(X_train, y_train)
test_pred = refined_champion.predict(X_test)

final = {
    "RMSE": np.sqrt(mean_squared_error(y_test, test_pred)),
    "MAE":  mean_absolute_error(y_test, test_pred),
    "R2":   r2_score(y_test, test_pred),
}
print(f"FINAL HOLD-OUT ({test_df.month.min()} .. {test_df.month.max()}):")
for k, v in final.items():
    print(f"  {k}: {v:,.3f}" if k == "R2" else f"  {k}: ${v:,.0f}")

signed = test_pred - y_test.values
print(f"\n  Mean signed error: ${signed.mean():,.0f}  (negative = conservative/under-predict)")
print(f"  Under-predicted: {(signed < 0).mean()*100:.1f}% of the future market")

FINAL HOLD-OUT (2025-01 .. 2026-10):
  RMSE: $65,537
  MAE: $48,789
  R2: 0.903

  Mean signed error: $-42,172  (negative = conservative/under-predict)
  Under-predicted: 84.9% of the future market


### E.4 Effect of the safety margin (lending view: keep conservative)

In [30]:
for m in [0.0, -0.03, 0.03]:
    adj = test_pred * (1 + m)
    over = (adj > y_test.values).mean() * 100
    print(f"margin {m:+.0%}:  MAE ${mean_absolute_error(y_test, adj):,.0f}   over-predicted {over:4.1f}% of flats")

margin +0%:  MAE $48,789   over-predicted 15.1% of flats
margin -3%:  MAE $63,407   over-predicted  7.0% of flats
margin +3%:  MAE $38,271   over-predicted 28.9% of flats


## Summary

- **Pipeline:** every learned step (impute, scale, one-hot, target-encode, log target) sits inside one `Pipeline`. Deterministic feature engineering runs before the split, and the notebook explains why that is safe. The `price_per_sqm` leak is avoided.
- **Champion:** HistGradientBoosting beat RandomForest and Ridge on time-series CV, with both a lower mean and a lower std RMSE (RMSE 52,840 vs 62,186 vs 72,331).
- **Tuning:** four controlled ablations, one variable each. The search was held to 24 fits (under the 50 cap) and every experiment is logged with its conclusion. Tuning lowered CV RMSE from 66,906 to 60,799. The raw target scored marginally better than the log target on CV (65,924); the log target was kept, for the reasons given in Experiment 4.
- **Failure:** the worst errors are systematic under-predictions of the sparse million-dollar tail and the rare flat types. Concrete fixes are proposed.
- **Decision:** over-prediction is the costlier error for the lender. The time-split model already errs on the conservative side, so the margin differs by use case: −3% for lending, +3% for buyer advice. That judgement ties directly to the Stage 1 cost structure.
- **Final hold-out (used once, 2025-01 → 2026-06 in the recorded run):** RMSE $64,269, MAE $49,418, **R² 0.903**. The model under-predicted 87.8% of flats, which is the conservative bias discussed in Part E.


## HDB valuation app (Gradio)

An interactive front end for the refined champion. Enter a flat and the app returns:
- the model's **estimated resale price**;
- the **Part E safety-margin views**: lender / collateral view at **−3%** and buyer-advice view at **+3%** (the margins tested in E.4);
- a table of the **most recent comparable sales** in the same town and flat type.

For valuations, the app refits the same tuned configuration on **all** transactions, including 2025 onwards, so it knows the latest price level. The hold-out metrics above were computed before this refit, on the frozen test set. The examples are the latest real sale for each town and flat type, so you can compare the estimate with the actual price in the comparables table. These sales are part of the app model's training data, so the comparison is a sanity check, not a test. The honest out-of-sample numbers are the hold-out metrics above.


In [ ]:
# ── HDB valuation app (Gradio) ─────────────────────────────────
import gradio as gr

# 1) Valuation model: same tuned configuration as the refined champion, refit on ALL transactions so the app
#    uses the latest price level. (Hold-out metrics above were computed before this, on the frozen test set.)
app_model = make_model(HGB(**best)).fit(data[FEATURES], data[TARGET])
LATEST_MONTH = data["month"].max()
print(f"App model refit on {len(data):,} transactions up to {LATEST_MONTH}.")

LENDER_MARGIN = -0.03          # Part E: lending / collateral view (keep conservative)
BUYER_MARGIN  = +0.03          # Part E: buyer-advice view (low end of the +3-5% correction)
HOLDOUT_LABEL = f"{test_df.month.min()} .. {test_df.month.max()}"
HOLDOUT_MAE   = final["MAE"]
HOLDOUT_UNDER = float((signed < 0).mean())

TOWNS       = sorted(data["town"].unique())
FLAT_TYPES  = sorted(data["flat_type"].unique())
FLAT_MODELS = sorted(data["flat_model"].unique())
STOREYS     = sorted(data["storey_range"].unique())
STREETS_BY_TOWN = {t: sorted(s.unique()) for t, s in data.groupby("town")["street_name"]}

# 2) Remaining lease. HDB leases run 99 years, but the data only gives the lease START YEAR. So we learn the
#    typical total lease length (months, counted from January of the start year) for each start year from the
#    data, then subtract the months elapsed up to the transaction month.
_lease_total = (data["remaining_lease_months"]
                + (data["txn_year"] - data["lease_commence_date"]) * 12
                + (data["txn_month_num"] - 1)).groupby(data["lease_commence_date"]).median()

def remaining_lease_str(lcd, year, month):
    total = float(_lease_total.get(lcd, 99 * 12))
    rem = int(round(total - ((year - lcd) * 12 + (month - 1))))
    rem = max(0, min(rem, 99 * 12))
    return f"{rem // 12} years {rem % 12:02d} months"     # same text format as the raw `remaining_lease` column

def build_row(town, flat_type, flat_model, storey_range, street_name,
              floor_area_sqm, lease_commence_date, txn_year, txn_month):
    """One raw row with the dataset's columns -> engineer() -> the FEATURES the pipeline expects."""
    lcd, y, m = int(lease_commence_date), int(txn_year), int(txn_month)
    raw = pd.DataFrame([{
        "month": f"{y:04d}-{m:02d}", "town": town, "flat_type": flat_type, "block": "",
        "street_name": street_name, "storey_range": storey_range,
        "floor_area_sqm": float(floor_area_sqm), "flat_model": flat_model,
        "lease_commence_date": lcd, "remaining_lease": remaining_lease_str(lcd, y, m),
    }])
    return engineer(raw)[FEATURES], raw.loc[0, "remaining_lease"]

COMP_COLS = ["month", "block", "street_name", "storey_range", "floor_area_sqm",
             "flat_model", "lease_commence_date", "resale_price"]

def peers_newest_first(town, flat_type):
    return (data[(data["town"] == town) & (data["flat_type"] == flat_type)]
            .sort_values("txn_date", ascending=False, kind="mergesort"))

def comparables(town, flat_type, n=10):
    c = peers_newest_first(town, flat_type).head(n)[COMP_COLS].copy()
    c["resale_price"] = c["resale_price"].map(lambda v: f"${v:,.0f}")
    return c

def value_flat(town, flat_type, flat_model, storey_range, street_name,
               floor_area_sqm, lease_commence_date, txn_year, txn_month):
    vals = [town, flat_type, flat_model, storey_range, street_name, floor_area_sqm,
            lease_commence_date, txn_year, txn_month]
    if any(v is None or v == "" for v in vals):
        return "Please fill in every field.", pd.DataFrame(columns=COMP_COLS)
    if not 1 <= int(txn_month) <= 12:
        return "Transaction month must be between 1 and 12.", pd.DataFrame(columns=COMP_COLS)
    if not 20 <= float(floor_area_sqm) <= 400:
        return "Floor area should be between 20 and 400 sqm.", pd.DataFrame(columns=COMP_COLS)

    X, lease_txt = build_row(*vals)
    est = float(app_model.predict(X)[0])
    lender, buyer = est * (1 + LENDER_MARGIN), est * (1 + BUYER_MARGIN)

    month_str = f"{int(txn_year):04d}-{int(txn_month):02d}"
    peers = data[(data["town"] == town) & (data["flat_type"] == flat_type)]
    recent = peers[peers["txn_date"] > peers["txn_date"].max() - pd.DateOffset(months=12)] if len(peers) else peers
    peer_txt = (f"Median of {len(recent):,} {flat_type} sales in {town} over the latest 12 months in the data: "
                f"**${recent['resale_price'].median():,.0f}**." if len(recent) else
                f"No recorded {flat_type} sales in {town}; treat this estimate with extra caution.")
    warn = (f"\n\n⚠ {month_str} is after the latest data month ({LATEST_MONTH}). Tree models hold the price level "
            f"flat beyond the data, so the estimate will not include any later market moves."
            if month_str > LATEST_MONTH else "")

    md = f"""### Estimated resale price: **${est:,.0f}**

| View | Margin | Figure |
|---|---|---|
| Model estimate | 0% | ${est:,.0f} |
| Lender / collateral view (Part E) | {LENDER_MARGIN:+.0%} | ${lender:,.0f} |
| Buyer-advice view (Part E) | {BUYER_MARGIN:+.0%} | ${buyer:,.0f} |

Remaining lease used: {lease_txt} · {peer_txt}

On the frozen hold-out ({HOLDOUT_LABEL}) the typical error was MAE **${HOLDOUT_MAE:,.0f}**, and the model
under-predicted **{HOLDOUT_UNDER:.0%}** of flats. In a rising market it errs on the conservative side, which is why the
lender view only trims further while the buyer view adds a small correction.{warn}"""
    return md, comparables(town, flat_type)

def streets_for(town, street):
    choices = STREETS_BY_TOWN.get(town, [])
    return gr.update(choices=choices, value=street if street in choices else (choices[0] if choices else None))

def _inputs_from_row(r):
    return [str(r["town"]), str(r["flat_type"]), str(r["flat_model"]), str(r["storey_range"]),
            str(r["street_name"]), float(r["floor_area_sqm"]), int(r["lease_commence_date"]),
            int(r["txn_year"]), int(r["txn_month_num"])]

def _latest_sale(town, flat_type):
    sub = peers_newest_first(town, flat_type)        # same order as the comparables table -> it is the top row
    return _inputs_from_row(sub.iloc[0]) if len(sub) else None

EXAMPLES = [e for e in (_latest_sale("ANG MO KIO", "3 ROOM"),
                        _latest_sale("PUNGGOL", "4 ROOM"),
                        _latest_sale("TAMPINES", "5 ROOM")) if e is not None]
DEFAULT = EXAMPLES[0] if EXAMPLES else _inputs_from_row(data.sort_values("txn_date").iloc[-1])

with gr.Blocks(title="HDB valuation") as demo:
    gr.Markdown("# HDB resale valuation\nTuned HistGradientBoosting pipeline (Group 7 capstone). "
                f"Trained on {len(data):,} Singapore HDB resale transactions ({data['month'].min()} .. {LATEST_MONTH}).")
    with gr.Tab("Valuation"):
        with gr.Row():
            with gr.Column():
                town       = gr.Dropdown(TOWNS, value=DEFAULT[0], label="Town")
                flat_type  = gr.Dropdown(FLAT_TYPES, value=DEFAULT[1], label="Flat type")
                flat_model = gr.Dropdown(FLAT_MODELS, value=DEFAULT[2], label="Flat model")
                storey     = gr.Dropdown(STOREYS, value=DEFAULT[3], label="Storey range")
                street     = gr.Dropdown(STREETS_BY_TOWN.get(DEFAULT[0], []), value=DEFAULT[4],
                                         label="Street name (filtered by town)", allow_custom_value=True)
                area       = gr.Number(value=DEFAULT[5], label="Floor area (sqm)")
                lcd        = gr.Number(value=DEFAULT[6], label="Lease commence year", precision=0)
                with gr.Row():
                    t_year  = gr.Number(value=DEFAULT[7], label="Transaction year", precision=0)
                    t_month = gr.Number(value=DEFAULT[8], label="Transaction month (1-12)", precision=0)
                btn = gr.Button("Estimate price", variant="primary")
            with gr.Column():
                out_md = gr.Markdown()
                out_comps = gr.Dataframe(label="Most recent comparable sales (same town + flat type)",
                                         interactive=False)
        inputs = [town, flat_type, flat_model, storey, street, area, lcd, t_year, t_month]
        town.change(streets_for, inputs=[town, street], outputs=street)
        btn.click(value_flat, inputs=inputs, outputs=[out_md, out_comps])
        gr.Examples(EXAMPLES, inputs=inputs, outputs=[out_md, out_comps], fn=value_flat, run_on_click=True,
                    cache_examples=False,
                    label="Latest real sale per town / flat type (its actual price is the top row of the comparables)")
    with gr.Tab("Model card"):
        gr.Markdown(f"**Final hold-out ({HOLDOUT_LABEL}, used once):** RMSE ${final['RMSE']:,.0f} · "
                    f"MAE ${final['MAE']:,.0f} · R² {final['R2']:.3f} (computed in this session)."
                    + ("" if TRAIN else "\n\nThe CV and ablation tables below are **recorded** from the full "
                       "training run (TRAIN = True)."))
        gr.Dataframe(cv_results.round(3), label="Part B: time-series CV model race", interactive=False)
        gr.Dataframe(ablation_log.round(2), label="Part C: controlled ablations (3-fold time-series CV)",
                     interactive=False)
    demo.load(value_flat, inputs=inputs, outputs=[out_md, out_comps])

demo.launch(share=True, debug=True)


App model refit on 243,182 transactions up to 2026-10.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1dd5ba703c645de2f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
